# Conditional independence test — fixed, refactored, and configurable

This replaces the seven near-identical `skew_*.ipynb` files. All shared logic lives in
**`ci_test.py`** (keep it in the same folder; on Colab upload it too). Generator
depth/width and the **data-generating model** are all just parameters now.

**What was fixed** (full story in the diagnosis report): the test itself is correct — with
the *true* generators the size is exactly nominal — but the learned generators were badly
under-trained and collapsed the conditional standard deviation to ~half of the truth, which
breaks double robustness and inflates the Type-I error. Fixes:

1. **Proper training** — `lr=5e-4` (was `5e-5`), no aggressive gradient clipping, and
   **early stopping** (deeper nets need more epochs; an under-converged net inflates size).
2. **Standardization actually applied** (per-variable, fit on the training fold).
3. **Selectable alignment penalty** — `mean` / `median` / `quantile` / `none` (see §1).
4. **Honest cross-fitting** — a fresh generator per fold (no warm-start leakage).
5. **Configurable depth/width**, **pluggable data-generating models** (§2), vectorized MMD,
   and the oracle kept as a built-in check.

## 0. Setup — every knob in one place
Field names in the comments are the matching names from the old notebooks' `param` dict.

In [ ]:
import numpy as np, torch
import ci_test as C

CONFIG = dict(C.DEFAULT_CONFIG)   # already carries all the fixes; override what you like
CONFIG.update(
    # --- architecture ---
    depth=2,             # hidden layers: 1/2/3 == old skew_1/2/3
    width=1024,          # hidden width                         (hidden_layer_size)
    noise_dim=5,         # latent noise dim                     (noise_dimension)

    # --- training (the main size fix) ---
    lr=5e-4,             # was 5e-5                              (G_lr)
    epochs=600,          # CAP; early stopping usually ends earlier (epochs_num)
    grad_clip=None,      # was 0.5 -> throttled learning
    M_train=20,          # draws per z for the training MMD      (M_train)

    # --- alignment penalty (see section 1) ---
    align_mode='quantile',   # 'none' | 'mean' | 'median' | 'quantile'
    lambda_align=0.5,        # penalty weight                   (lambda_median)
    taus=(0.1, 0.5, 0.9),    # quantiles used when align_mode='quantile'
    align_samples=64,        # draws per z for the penalty       (median_samples)

    # --- test statistic / bootstrap ---
    n_folds=2,           # cross-fitting folds J                 (k_value)
    M_test=100,          # draws per z for the statistic         (m_value)
    n_boot=1000,         # wild-bootstrap replicates B           (j_value)
    standardize=True,
)
CONFIG

## 1. Alignment penalty — choose the mode

The extension idea is to add an alignment penalty on top of the conditional-MMD loss.
`align_mode` selects which summary of P(target|Z) is pulled toward the data:

| mode | penalty | what it pins | note |
|------|---------|--------------|------|
| `none` | – | nothing | pure MMD, as in the original paper |
| `mean` | MSE(E[Ŷ\|Z], Y) | the conditional **mean** | the tutorial's idea |
| `median` | pinball τ=0.5 | the conditional **median** | the original notebook's penalty |
| `quantile` | pinball over `taus` | center **and spread** | recommended (fixes the collapse) |

On the skewed data the failure was under-dispersion, so `mean`/`median` (center-only) help
less than `quantile`, which also matches the 0.1 and 0.9 conditional quantiles.

## 2. Data-generating model (DGP) — choose or add one

The data-generating model is pluggable. Five are built in, covering the DGPs used across
Simulations 4–8:

| name | from | description |
|------|------|-------------|
| `'skew'` | Sim 6/8 | skewed lognormal noise, heteroscedastic spread, **nonlinear** means |
| `'skew_linear'` | Sim 8 §1.3 | same, but **linear** means `mX=0.8z, mY=-0.6z` |
| `'gaussian'` | Sim 4/5 (§4.1) | `Z=e3, Y=Z+e1, X=Z+δe1+(1-δ)e2` — Gaussian / linear |
| `'heteroskedastic'` | Sim 5 §3.1 | `Y=Z+εy, X=σ(Z)εx, σ=0.3+1.2\|Z\|` — zero-mean, Z-dependent variance |
| `'student_t'` | Sim 5 §4 | `mX=mY=Z, s=0.5+\|Z\|`, standardized Student-t noise (heavy-tailed) |

Knobs go through `data_kwargs`: `'skew'`/`'skew_linear'` take `dx, dy, dz, nstd, dist_z, alpha_x`;
the others take `dz, alpha_x` (`'student_t'` also `df`, default 3). Pick one here; every
experiment below passes `dgp=DGP_NAME`.

In [ ]:
for name, d in C.DGPS.items():
    print(f'{name:9s}: {d.description}')

DGP_NAME = 'skew'      # <- change to 'gaussian' (Section 4.1) or your own registered DGP


## 3. Oracle sanity check — always run first

Swap the learned generators for the **true** conditionals of the chosen DGP. This must sit
at the nominal level (≈0.05 at α=0.05). If it does but the learned test below doesn't, the
gap is purely generator quality — not a bug in the test. Keep this as your unit test.

In [ ]:
C.run_experiment(n=400, hypothesis='H0', n_rep=100, config=CONFIG,
                 dgp=DGP_NAME, oracle=True, n_jobs=-1)


## 4. Empirical size (Type-I error) with the learned generators
Under H0 the rejection rate should be close to the nominal level.

In [ ]:
C.run_experiment(n=400, hypothesis='H0', n_rep=100, config=CONFIG,
                 dgp=DGP_NAME, oracle=False, n_jobs=-1)


## 5. Power (H1)
Dependence strength grows with `alpha_x`. Sweep it to trace a power curve.

In [ ]:
for a in [0.05, 0.10, 0.15, 0.20, 0.25]:
    r = C.run_experiment(n=400, hypothesis='H1', n_rep=100, config=CONFIG, dgp=DGP_NAME,
                         oracle=False, data_kwargs=dict(alpha_x=a), n_jobs=-1, verbose=False)
    print(f'alpha_x={a:.2f}  power@0.05={r["rejection"][0.05]:.3f}')


## 6. Size check across all built-in DGPs (Simulations 4–8)

Confirm the learned test controls its size on every built-in DGP. All should land near the
nominal level; the heavy-tailed `'student_t'` is the hardest — raise `epochs`/`n_rep` if it
looks borderline.

In [ ]:
for name in ['skew', 'skew_linear', 'gaussian', 'heteroskedastic', 'student_t']:
    print(f'{name:16s}', end=' ')
    C.run_experiment(n=400, hypothesis='H0', n_rep=100, config=CONFIG,
                     dgp=name, oracle=False, n_jobs=-1)


A power example on the paper's Section 4.1 `'gaussian'` model:

In [ ]:
C.run_experiment(n=400, hypothesis='H1', n_rep=100, config=CONFIG, dgp='gaussian',
                 oracle=False, data_kwargs=dict(alpha_x=0.15), n_jobs=-1)


## 7. Register your own DGP

A DGP bundles two functions: how to draw `(X, Y, Z)`, and how to draw from the **true**
conditionals `P(X|Z)`, `P(Y|Z)` (used only by the oracle check). Register it, then pass
`dgp='my_dgp'` anywhere above.

In [ ]:
def my_sample(n, hypothesis, dz=1, device=None, **_):
    Z  = torch.randn(n, dz, device=device)
    e1 = torch.randn(n, 1, device=device)
    e2 = torch.randn(n, 1, device=device)
    Y  = torch.sin(Z[:, :1]) + e1
    X  = torch.sin(Z[:, :1]) + (e1 if hypothesis == 'H1' else e2)   # dependent under H1
    return X, Y, Z

def my_oracle(Z, m, device=None, **_):
    mu = torch.sin(Z[:, :1]).unsqueeze(1)                            # true E[.|Z]
    X = mu + torch.randn(Z.shape[0], m, 1, device=device)
    Y = mu + torch.randn(Z.shape[0], m, 1, device=device)
    return X, Y

C.register_dgp(C.DGP('my_dgp', my_sample, my_oracle, 'sin mean, gaussian noise (demo)'))
C.run_experiment(n=400, hypothesis='H0', n_rep=50, config=CONFIG, dgp='my_dgp',
                 oracle=True, n_jobs=-1)


## 8. Compare alignment modes / architectures (optional)

In [ ]:
for mode in ['none', 'mean', 'median', 'quantile']:
    cfg = dict(CONFIG); cfg['align_mode'] = mode
    print(f'align_mode={mode:9s}', end='  ')
    C.run_experiment(n=400, hypothesis='H0', n_rep=100, config=cfg, dgp=DGP_NAME, oracle=False, n_jobs=-1)

for depth in [1, 2, 3]:
    cfg = dict(CONFIG); cfg['depth'] = depth
    print(f'depth={depth}', end='  ')
    C.run_experiment(n=400, hypothesis='H0', n_rep=100, config=cfg, dgp=DGP_NAME, oracle=False, n_jobs=-1)


---
### Tips
* `n_jobs=-1` uses all CPU cores (joblib). On a GPU leave `n_jobs=1` (`prefer_gpu=True` by default).
* Quick smoke run: drop `epochs`, `n_rep`, `n_boot`, `M_test` — expect noisier numbers.
* Every default lives in `C.DEFAULT_CONFIG` inside `ci_test.py`, each with a one-line comment and its old `param` name.
* See **`NOTE_for_students.pdf`** for a short walk-through of what changed and how to use this.